# Tiny Shakespeare Text Generator — End-to-End Project

**Goal:** train a small decoder-only Transformer from scratch, using a custom byte-level BPE tokenizer trained on Tiny Shakespeare.

**Configuration:** vocabulary 1,000; 2 Transformer blocks; 2 attention heads; embedding size 128; context 64; dropout 0.1; 5 epochs; batch size 32; AdamW; learning rate 1e-3.

> Runtime note: this notebook uses the complete tokenized dataset and defines an epoch as one pass through a fixed number of randomly sampled training windows (`len(train_ids) // (batch_size * block_size)`). Runtime depends on the free GPU assigned by Colab. The one-hour target is a goal, not a guarantee. If needed, reduce `MAX_STEPS_PER_EPOCH` while clearly documenting the change.

Run cells from top to bottom in Google Colab. Select **Runtime → Change runtime type → T4 GPU** (or another available GPU).

## 1. Setup and check the GPU

No external tokenization libraries are used. The tokenizer below is implemented with Python's standard library.

In [ ]:
import os, re, math, time, json, random, urllib.request
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device:", device)
if device == "cpu":
    print("WARNING: GPU not detected. Enable a GPU in Runtime > Change runtime type.")

## 2. Download Tiny Shakespeare

The source is Andrej Karpathy's public Tiny Shakespeare text file.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "input.txt"

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

print("Characters:", len(text))
print("Preview:\\n", text[:500])

## 3. Implement byte-level BPE from scratch

This tokenizer starts with the 256 possible byte values and learns frequent adjacent-byte/token merges until the vocabulary reaches 1,000 entries. A standard-library regular expression separates text into word, punctuation, and whitespace pieces; BPE merges are learned from piece frequencies.

This is a compact educational BPE implementation, not a reproduction of every detail of GPT-2's tokenizer.

In [ ]:
class SimpleBPE:
    def __init__(self, vocab_size=1000):
        if vocab_size < 256:
            raise ValueError("Byte-level BPE requires vocab_size >= 256")
        self.vocab_size = vocab_size
        self.merges = []          # ordered (left_id, right_id) pairs
        self.merge_ranks = {}     # pair -> rank
        self.pattern = re.compile(r"\\w+|[^\\w\\s]+|\\s+", re.UNICODE)

    def _pieces(self, text):
        return self.pattern.findall(text)

    def train(self, text):
        counts = Counter(self._pieces(text))
        sequences = {
            tuple(piece.encode("utf-8")): freq
            for piece, freq in counts.items()
        }

        num_merges = self.vocab_size - 256
        for merge_i in range(num_merges):
            pair_counts = Counter()
            for seq, freq in sequences.items():
                for j in range(len(seq) - 1):
                    pair_counts[(seq[j], seq[j+1])] += freq

            if not pair_counts:
                break
            best_pair, best_count = pair_counts.most_common(1)[0]
            if best_count < 2:
                break

            new_id = 256 + len(self.merges)
            self.merges.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merges) - 1

            updated = {}
            for seq, freq in sequences.items():
                out = []
                j = 0
                while j < len(seq):
                    if j < len(seq)-1 and (seq[j], seq[j+1]) == best_pair:
                        out.append(new_id)
                        j += 2
                    else:
                        out.append(seq[j])
                        j += 1
                key = tuple(out)
                updated[key] = updated.get(key, 0) + freq
            sequences = updated

            if (merge_i + 1) % 100 == 0:
                print(f"Learned {merge_i+1}/{num_merges} merges; pair count={best_count}")

        self.actual_vocab_size = 256 + len(self.merges)
        print("Tokenizer vocabulary size:", self.actual_vocab_size)
        return self

    def encode_piece(self, piece):
        ids = list(piece.encode("utf-8"))
        while len(ids) >= 2:
            best_rank = None
            best_pair = None
            for j in range(len(ids)-1):
                pair = (ids[j], ids[j+1])
                rank = self.merge_ranks.get(pair)
                if rank is not None and (best_rank is None or rank < best_rank):
                    best_rank, best_pair = rank, pair
            if best_pair is None:
                break

            merged_id = 256 + best_rank
            out = []
            j = 0
            while j < len(ids):
                if j < len(ids)-1 and (ids[j], ids[j+1]) == best_pair:
                    out.append(merged_id)
                    j += 2
                else:
                    out.append(ids[j])
                    j += 1
            ids = out
        return ids

    def encode(self, text):
        result = []
        for piece in self._pieces(text):
            result.extend(self.encode_piece(piece))
        return result

    def decode(self, ids):
        byte_chunks = []
        for token_id in ids:
            token_id = int(token_id)
            if token_id < 256:
                byte_chunks.append(bytes([token_id]))
            else:
                left, right = self.merges[token_id - 256]
                # Expand merged tokens recursively using their construction.
                byte_chunks.append(self._token_bytes(token_id))
        return b"".join(byte_chunks).decode("utf-8", errors="replace")

    def _token_bytes(self, token_id):
        if token_id < 256:
            return bytes([token_id])
        left, right = self.merges[token_id - 256]
        return self._token_bytes(left) + self._token_bytes(right)

    def save(self, path="bpe_tokenizer.json"):
        payload = {"vocab_size": self.vocab_size, "merges": self.merges}
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        obj = cls(payload["vocab_size"])
        obj.merges = [tuple(x) for x in payload["merges"]]
        obj.merge_ranks = {pair:i for i, pair in enumerate(obj.merges)}
        obj.actual_vocab_size = 256 + len(obj.merges)
        return obj


tokenizer = SimpleBPE(vocab_size=1000)
tokenizer.train(text)
tokenizer.save()
sample_ids = tokenizer.encode("To be, or not to be.")
print("Encoded IDs:", sample_ids[:30])
print("Decoded:", tokenizer.decode(sample_ids))

## 4. Encode the full dataset and create the 90/10 split

The split is chronological: the first 90% is training data and the final 10% is validation data. This avoids randomly mixing future text into training.

In [ ]:
all_ids = tokenizer.encode(text)
data = torch.tensor(all_ids, dtype=torch.long)
split_idx = int(0.90 * len(data))
train_ids = data[:split_idx]
val_ids = data[split_idx:]

print(f"Total tokens: {len(data):,}")
print(f"Train tokens: {len(train_ids):,} ({len(train_ids)/len(data):.1%})")
print(f"Validation tokens: {len(val_ids):,} ({len(val_ids)/len(data):.1%})")
print("Tokenizer vocab:", tokenizer.actual_vocab_size)

## 5. Set hyperparameters and batch creation

In [ ]:
# Required project settings
BATCH_SIZE = 32
BLOCK_SIZE = 64
MAX_EPOCHS = 5
LEARNING_RATE = 1e-3
DROPOUT = 0.1
N_EMBED = 128
N_HEAD = 2
N_LAYER = 2

# One pass through approximately non-overlapping windows per epoch.
STEPS_PER_EPOCH = max(1, len(train_ids) // (BATCH_SIZE * BLOCK_SIZE))
# Optional safety cap if a slower GPU is assigned. Leave as None for full epochs.
MAX_STEPS_PER_EPOCH = None
if MAX_STEPS_PER_EPOCH is not None:
    STEPS_PER_EPOCH = min(STEPS_PER_EPOCH, MAX_STEPS_PER_EPOCH)

def get_batch(split):
    source = train_ids if split == "train" else val_ids
    ix = torch.randint(len(source) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([source[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([source[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)

print("Steps per epoch:", STEPS_PER_EPOCH)
print("Total optimizer steps:", STEPS_PER_EPOCH * MAX_EPOCHS)

## 6. Build the decoder-only Transformer

The model uses causal (masked) self-attention so a token can only attend to itself and earlier tokens. The output head predicts the next token at each position.

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_EMBED, head_size, bias=False)
        self.query = nn.Linear(N_EMBED, head_size, bias=False)
        self.value = nn.Linear(N_EMBED, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        weights = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        weights = F.softmax(weights, dim=-1)
        weights = self.dropout(weights)
        return self.dropout(weights @ self.value(x))


class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = N_EMBED // N_HEAD
        self.heads = nn.ModuleList([Head(head_size) for _ in range(N_HEAD)])
        self.proj = nn.Linear(N_EMBED, N_EMBED)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_EMBED, 4 * N_EMBED),
            nn.ReLU(),
            nn.Linear(4 * N_EMBED, N_EMBED),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = MultiHeadAttention()
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(N_EMBED)
        self.ln2 = nn.LayerNorm(N_EMBED)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, N_EMBED)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, N_EMBED)
        self.blocks = nn.Sequential(*[TransformerBlock() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBED)
        self.lm_head = nn.Linear(N_EMBED, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding(idx)
        pos = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(B*T, -1), targets.reshape(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx


vocab_size = tokenizer.actual_vocab_size
model = TinyGPT(vocab_size).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. Train for five epochs and log losses

Validation loss is measured without gradient updates. We average several validation batches for a more stable estimate.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
train_losses, val_losses = [], []

@torch.no_grad()
def estimate_loss(eval_batches=100):
    model.eval()
    results = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_batches)
        for k in range(eval_batches):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        results[split] = losses.mean().item()
    model.train()
    return results

start_time = time.time()
for epoch in range(MAX_EPOCHS):
    model.train()
    running = 0.0
    for step in range(STEPS_PER_EPOCH):
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        running += loss.item()

    train_epoch_loss = running / STEPS_PER_EPOCH
    metrics = estimate_loss(eval_batches=50)
    train_losses.append(train_epoch_loss)
    val_losses.append(metrics["val"])
    elapsed = (time.time() - start_time) / 60
    print(
        f"Epoch {epoch+1}/{MAX_EPOCHS} | "
        f"train loss: {train_epoch_loss:.4f} | "
        f"validation loss: {metrics['val']:.4f} | "
        f"elapsed: {elapsed:.1f} min"
    )

total_minutes = (time.time() - start_time) / 60
print(f"Training finished in {total_minutes:.2f} minutes")

## 8. Plot and save the loss curves

In [ ]:
epochs = range(1, len(train_losses) + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, marker="o", label="Training loss")
plt.plot(epochs, val_losses, marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Tiny Shakespeare Transformer: Loss by Epoch")
plt.xticks(list(epochs))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("loss_plot.png", dpi=160)
plt.show()

## 9. Generate 100-token samples with temperature sampling

Temperature controls randomness. Lower values make sampling more conservative; higher values make it more varied. The prompt is encoded with the same BPE tokenizer.

In [ ]:
def generate_sample(prompt="ROMEO:", temperature=0.8, new_tokens=100):
    prompt_ids = tokenizer.encode(prompt)
    if not prompt_ids:
        prompt_ids = [ord(" ")]
    context = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    model.eval()
    with torch.no_grad():
        output = model.generate(context, max_new_tokens=new_tokens, temperature=temperature)
    model.train()
    return tokenizer.decode(output[0].tolist())

for temp in [0.7, 0.8, 1.0]:
    print("\\n" + "="*25 + f" temperature={temp} " + "="*25)
    print(generate_sample("ROMEO:", temperature=temp, new_tokens=100))

In [ ]:
# Save the generated examples so they can be included in the submission.
with open("generated_samples.txt", "w", encoding="utf-8") as f:
    for temp in [0.7, 0.8, 1.0]:
        sample = generate_sample("ROMEO:", temperature=temp, new_tokens=100)
        f.write(f"\\n===== Temperature {temp} =====\\n")
        f.write(sample + "\\n")
print("Saved generated_samples.txt")

## 10. Save model, tokenizer, and run metadata

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "vocab_size": vocab_size,
    "config": {
        "n_layer": N_LAYER, "n_head": N_HEAD, "n_embed": N_EMBED,
        "block_size": BLOCK_SIZE, "dropout": DROPOUT
    }
}, "tiny_gpt.pt")

tokenizer.save("bpe_tokenizer.json")
with open("training_metrics.json", "w") as f:
    json.dump({
        "train_losses": train_losses,
        "validation_losses": val_losses,
        "training_minutes": total_minutes,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "vocab_size": vocab_size
    }, f, indent=2)

print("Saved: tiny_gpt.pt, bpe_tokenizer.json, training_metrics.json, loss_plot.png")

## 11. One-page findings summary (fill in after running)

Copy the measured values printed by your notebook. Do not invent results. Include one or two generated samples in your final submission.

**Project:** Tiny Shakespeare Text Generation with a Custom BPE Tokenizer and Decoder-only Transformer

**Objective.** Build a compact autoregressive language model that learns character dialogue and Shakespearean-style text patterns from Tiny Shakespeare.

**Method.** A byte-level BPE tokenizer was implemented using Python's standard library. It starts with 256 byte tokens and learns frequent adjacent-token merges up to a target vocabulary of 1,000. The tokenized corpus was split chronologically into 90% training and 10% validation data. A decoder-only Transformer with 2 blocks, 2 attention heads, 128-dimensional embeddings, context length 64 and dropout 0.1 was trained for 5 epochs using batch size 32, AdamW and learning rate 0.001.

**Results.** Training took `___` minutes. Training loss changed from `___` to `___`; validation loss changed from `___` to `___`. Insert the saved loss plot (`loss_plot.png`) and describe whether the two curves suggest learning, underfitting or overfitting.

**Text generation.** The model generated 100 new tokens using temperature sampling. Compare outputs at temperatures 0.7, 0.8 and 1.0. Lower temperature should generally produce more predictable text, while higher temperature tends to increase variation. Include a short representative output.

**Limitations.** This is a small model trained for only five epochs on a small dataset. It can imitate surface patterns but may produce incoherent dialogue, inconsistent characters or grammatical errors. It does not understand Shakespeare or guarantee historically accurate language.

**Conclusion.** State whether the model learned useful statistical patterns, what the loss curves show, and what you would improve next (more training, larger context, learning-rate scheduling, or a larger model).

## 12. Submission checklist

- [ ] Notebook runs from top to bottom in Google Colab.
- [ ] GPU type and runtime are recorded.
- [ ] BPE vocabulary size and merge count are reported.
- [ ] Train/validation split is 90/10.
- [ ] Five epochs completed (or any runtime cap is disclosed).
- [ ] Training and validation losses are logged.
- [ ] `loss_plot.png` is saved.
- [ ] Three temperature samples are included.
- [ ] `tiny_gpt.pt`, `bpe_tokenizer.json`, and `training_metrics.json` are saved.
- [ ] One-page findings summary is completed with actual results.
- [ ] Notebook and supporting files are uploaded to GitHub.